In [ ]:
import os
import sys


import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


from pynaowee.utils import download_file
from pynaowee.utils import display_categorical_values

%matplotlib inline 

In [ ]:
DATASET_DIR = "./datasets/"
DATALAKEHOUSE_DIR = "./datalakehouse/"

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(DATALAKEHOUSE_DIR, exist_ok=True)

In [ ]:
DUCKDB_FILE = os.path.join(DATALAKEHOUSE_DIR, "datalakehouse.duckdb")

In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/daramireh/simonBolivarCienciaDatos/refs/heads/main/Student_Performance.csv"
DATASET_FILE = os.path.join(DATASET_DIR, "student_performance.csv")

if not os.path.exists(DATASET_FILE):
    download_file(DATASET_URL, DATASET_FILE)

In [ ]:
DUCKDB_FILE = os.path.join(DATALAKEHOUSE_DIR, "datalakehouse.duckdb")

with duckdb.connect(DUCKDB_FILE) as conn:
    
    conn.execute("CREATE SCHEMA IF NOT EXISTS s10_raw;")
    conn.execute("DROP TABLE IF EXISTS s10_raw.student_performance;")
    conn.execute(f"""
        CREATE TABLE IF NOT EXISTS s10_raw.student_performance AS 
        SELECT * FROM read_csv_auto('{DATASET_FILE}');
    """)
    print("Table s10_raw.student_performance created in DuckDB database.")


In [ ]:

df_student_performance = None
with duckdb.connect(DUCKDB_FILE) as conn:
    df_student_performance = conn.execute("SELECT * FROM s10_raw.student_performance;").df()

In [ ]:
# transform all column to snake_case 
df_student_performance.columns = [
    col.lower().replace(" ", "_").strip() for col in df_student_performance.columns
]

In [ ]:
df_student_performance.info()

In [ ]:
df_student_performance.head()

In [ ]:
df_student_performance.describe(include='all')

- Exiten cinco variables numéricas y solo una categórica binaria. 
- La variable objetivo llama **performance index**.



In [ ]:
from ydata_profiling import ProfileReport

os.makedirs("eda-reports", exist_ok=True)
ProfileReport(
    df_student_performance,
    explorative=True,
    title="Sudent Math Performance Report",
).to_file("eda-reports/student-performance-report.html")

#### Conclusiones del reporte generado.

```plain
Dataset statistics
Number of variables	6
Number of observations	10000
Missing cells	0
Missing cells (%)	0.0%
Duplicate rows	126
Duplicate rows (%)	1.3%
Total size in memory	400.5 KiB
Average record size in memory	41.0 B
```

1. No se presentan datos nulos por lo tanto no hay que aplicar operaciones como eliminar las filas con nulos, o rellenar esas celdas faltantes. 
2. El dataset presenta un 126 datos nulos que se recomienda que sean removidos. 
3. A pesar de contener gran numero de filas, el hecho que sean solo seis columnas y que no sean de tipo texto, hace que el consumo en memoria del dataset sea pequeño. 
Por lo tanto, no se requieren el uso de herramieintas complejas de BigData, y para el caso de machine learing, el uso de modelos demasiado complejos puede llevar al sobreajuste *overfitting*.


###### Analisis de las variables.

1. La variable de horas estudiadas tiene una distribución uniforme. 
Los valores se encuentran en rangos "normales". 

![hours-studied](media/caso2/studied-hours.png)

2. La variable previus-scores tiene una distribución uniforme dentro de rangos que tienen sentido en el contexto. 
Se presentan unos picos en algunos que están uniformemente distribuidos. 
Una elección de `bins` más apropiada mostraría una distribición más similar a la distribuciópn uniforme.

![previous-scores](media/caso2/studied-hours.png)

3. La variables  `extracurricular_activities` es categórica y binaria. La distribución de los datos es balanceada. 

![extracurricular-activities](media/caso2/extracurricular-activities.png)

4. La variable de horas de sueño tiene una distribución uniforme. 
Los valores se encuentran en rangos "normales". 

![sleep-hours](media/caso2/sleep-hours.png)


5. La variable `sample_question_papers_practiced` es categórica y binaria. La distribución de los datos es balanceada. 

![question-practiced](media/caso2/question-practiced.png)


6. La variable Objetivo sigue una distribución casi normal. No se presentan valores extremos. 

![performance-index](media/caso2/performence-index.png)




In [ ]:
df_student_performance.drop_duplicates(inplace=True)
df_student_performance.reset_index(drop=True, inplace=True)

#### Relación entre las variables

In [ ]:
plt.figure(figsize=(12, 10))

# Configurar el estilo visual de seaborn (opcional, pero se ve bien)
sns.set_style("whitegrid")

# Crear el pairplot
# 'hue' define la variable categórica para colorear los puntos
# 'palette' define la paleta de colores a usar
# 'diag_kind' define qué tipo de gráfico usar para las distribuciones univariantes (diagonal)
df_student_performance_sample = df_student_performance.sample(n=500, random_state=42)   
sns.pairplot(df_student_performance_sample, 
             hue='extracurricular_activities', 
             palette='bright', 
             diag_kind='kde') # kde muestra una curva de densidad en la diagonal en lugar de un histograma

# Añadir un título general a la figura
plt.suptitle('Gráficos de Dispersión de Interacciones con Actividades Extracurriculares', y=1.02)

# Mostrar el gráfico
plt.show()

#### Grafico de correlación. 

In [ ]:
# calcular grafico de correlacion

correlation_matrix = df_student_performance.corr()
# imprtimir los numeros de correlacion

print(correlation_matrix)

![correlation-matrix](media/caso2/correlation-matrix.png)

1. Relación Dominante: previous_scores y performance_index

    Correlación Fuerte y Positiva (0.915): Esta es, con diferencia, la relación más importante del conjunto de datos. Existe una correlación lineal extremadamente fuerte entre las puntuaciones obtenidas en exámenes anteriores (previous_scores) y el índice de rendimiento final (performance_index).
    Conclusión: El predictor más fiable del rendimiento futuro de un estudiante es su rendimiento pasado. Si un estudiante tiene puntuaciones altas previas, es casi seguro que tendrá un alto índice de rendimiento.

2. Relación Moderada: hours_studied y performance_index

    Correlación Moderada y Positiva (0.375): Hay una relación positiva, pero mucho más débil que la anterior, entre las horas estudiadas (hours_studied) y el performance_index.
    Conclusión: Estudiar más horas ayuda a mejorar el rendimiento, pero su influencia es significativamente menor que las puntuaciones previas del estudiante. Otros factores no medidos, o la calidad del estudio, podrían ser relevantes aquí.

3. Ausencia de Correlación Significativa (Variables con correlación cercana a cero)
Varias variables muestran una correlación prácticamente nula con el performance_index y entre sí:

    sleep_hours (0.050): Las horas de sueño tienen un impacto insignificante en el índice de rendimiento.
    sample_question_papers_practiced (0.043): Practicar exámenes de muestra tiene un impacto muy limitado en el rendimiento final.
    extracurricular_activities (0.026): Participar en actividades extracurriculares no muestra una correlación medible con el rendimiento académico en este dataset.

4. Relaciones entre las Variables Predictoras

    Es interesante notar que hours_studied y previous_scores tienen una correlación casi nula entre sí (-0.010). Esto sugiere que la cantidad de horas que un estudiante estudia actualmente no está relacionada con las calificaciones que obtuvo históricamente antes de este período de estudio.

In [ ]:
from scipy import stats

grupo_con_actividades = df_student_performance[df_student_performance['extracurricular_activities'] == True]['performance_index']
grupo_sin_actividades = df_student_performance[df_student_performance['extracurricular_activities'] == False]['performance_index']

# Opcional: Verificar las medias antes del test
print(f"Media con actividades: {grupo_con_actividades.mean():.2f}")
print(f"Media sin actividades: {grupo_sin_actividades.mean():.2f}\n")


# --- 3. Realizar el Test t de Student ---

# Usamos ttest_ind (independent samples t-test)
# 'equal_var=False' se usa a menudo por precaución (Welch's t-test) si las varianzas son diferentes
t_statistic, p_value = stats.ttest_ind(grupo_con_actividades, 
                                       grupo_sin_actividades, 
                                       equal_var=False, 
                                       alternative='greater') # Especificamos 'greater' para H1 unilateral

print(f"Estadístico t: {t_statistic:.4f}")
print(f"Valor p (p-value): {p_value:.4f}")

# --- 4. Interpretar el resultado ---

alpha = 0.05 # Nivel de significancia común

print(f"\nNivel de significancia (alpha): {alpha}")

if p_value < alpha:
    print("\nCONCLUSION: Se rechaza la Hipótesis Nula (H0).")
    print("Hay evidencia estadística significativa para sugerir que \n" + 
          "los estudiantes con actividades extracurriculares tienen un MAYOR índice de rendimiento promedio.")
else:
    print("\nCONCLUSION: No se rechaza la Hipótesis Nula (H0).")
    print("No hay evidencia estadística suficiente para concluir que \n" + 
          "los estudiantes con actividades extracurriculares tienen un mejor índice de rendimiento que los que no las tienen.")

In [ ]:
with duckdb.connect(DUCKDB_FILE) as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS s11_curated;")
    conn.execute("DROP TABLE IF EXISTS s11_curated.student_performance_curated;")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS s11_curated.student_performance_curated AS 
        SELECT * FROM df_student_performance;
    """)
    print("Table s11_curated.student_performance_curated created in DuckDB database.")